# Book Database Analysis Using SQL

**Data Analyst Project | January 2026**

## Project Overview

The COVID-19 pandemic accelerated the adoption of digital reading platforms as consumers shifted toward at-home entertainment. This project analyzes a database containing information about books, authors, publishers, ratings, and customer reviews to uncover patterns in reader engagement and publishing performance.

## Objectives

The objective of this analysis is to generate insights that could support the development of a new book recommendation platform by:

1. Identifying the volume of books published after January 1, 2000.
2. Evaluating reader engagement through ratings and reviews.
3. Determining which publishers produce the highest number of substantial books (more than 50 pages).
4. Identifying authors with the strongest reader ratings.
5. Analyzing review activity among highly engaged users.

The analysis was performed using SQL queries to extract, aggregate, and interpret data from a relational database containing books, authors, publishers, ratings, and reviews.


## Database Overview


In [2]:
# Database connection omitted from the public version
# Credentials removed for security purposes

## Data Preview

Before performing the analysis, the first rows of each table were reviewed to understand the database structure, verify available fields, and identify relationships between books, authors, publishers, ratings, and reviews.

In [3]:
# Preview database tables
tables = [
    'books',
    'authors',
    'publishers',
    'ratings',
    'reviews'
]

for table in tables:
    print(f"\nFirst rows from the {table} table:")
    display(
        pd.read_sql(
            f"SELECT * FROM {table} LIMIT 5;",
            con=engine
        )
    )



First rows from the books table:


,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268



First rows from the authors table:


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd



First rows from the publishers table:


,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company



First rows from the ratings table:


,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2



First rows from the reviews table:


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


The database consists of five related tables:

- `books` contains book information and links to authors and publishers.
- `authors` contains author names and identifiers.
- `publishers` contains publisher information.
- `ratings` stores user ratings for books.
- `reviews` contains written reviews submitted by users.

The relationships between tables are primarily established through the `book_id` field, allowing book information to be combined with ratings, reviews, authors, and publisher data. Together, these tables provide the necessary information to analyze publishing activity, reader engagement, and author performance.

## SQL Analysis

This section answers the main business questions using SQL queries. Each query extracts and summarizes information from the relational database to support product and market insights for a digital book platform.

### Modern Book Catalog Analysis
To understand the availability of modern titles in the catalog, the number of books published after January 1, 2000 was calculated.

In [4]:
# Count books published after January 1, 2000
query_1 = """
SELECT COUNT(*) AS books_after_2000
FROM books
WHERE publication_date > '2000-01-01';
"""

books_after_2000 = pd.read_sql(query_1, con=engine)
books_after_2000

,books_after_2000
0,819


A total of **819 books** were published after January 1, 2000.

This indicates that a substantial portion of the catalog consists of modern publications. Since the analysis focuses on a digital reading platform, the strong representation of contemporary titles provides valuable insight into recent publishing trends and reader interests.

### Reader Engagement and Book Ratings

This query calculates the number of written reviews and the average user rating for each book. These metrics help evaluate both reader engagement and overall audience perception.

In [5]:
# Calculate review counts and average ratings for each book
query_2 = """
SELECT
    b.book_id,
    b.title,
    COUNT(DISTINCT r.review_id) AS review_count,
    ROUND(AVG(rt.rating), 2) AS average_rating
FROM books AS b
LEFT JOIN reviews AS r ON b.book_id = r.book_id
LEFT JOIN ratings AS rt ON b.book_id = rt.book_id
GROUP BY
    b.book_id,
    b.title
ORDER BY
    review_count DESC;
"""

reviews_ratings_by_book = pd.read_sql(query_2, con=engine)
reviews_ratings_by_book.head()

,book_id,title,review_count,average_rating
0,948,Twilight (Twilight #1),7,3.66
1,963,Water for Elephants,6,3.98
2,734,The Glass Castle,6,4.21
3,302,Harry Potter and the Prisoner of Azkaban (Harr...,6,4.41
4,695,The Curious Incident of the Dog in the Night-Time,6,4.08


The books with the highest review activity received between 6 and 7 written reviews. While *Twilight* generated the largest number of reviews (7), *Harry Potter and the Prisoner of Azkaban* achieved the highest average rating (4.41) among the most-reviewed books.

The relatively low number of written reviews suggests that users may be more likely to provide ratings than written feedback.

### Leading Publisher by Book Volume

To focus on substantial publications rather than short booklets or pamphlets, this analysis identifies the publisher that has released the highest number of books containing more than 50 pages.

In [6]:
# Identify the publisher with the largest number of books over 50 pages
query_3 = """
SELECT
    p.publisher,
    COUNT(b.book_id) AS book_count
FROM books b
JOIN publishers p
    ON b.publisher_id = p.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher
ORDER BY book_count DESC
LIMIT 1;
"""

leading_publisher = pd.read_sql(query_3, con=engine)
leading_publisher

,publisher,book_count
0,Penguin Books,42


Penguin Books published the highest number of books with more than 50 pages, totaling 42 titles.

This suggests that Penguin Books maintains a strong presence within the catalog and contributes a significant share of substantial publications.

### Top-Rated Author Analysis

This analysis identifies the author with the highest average book rating while considering only books that received at least 50 ratings. Applying this threshold helps ensure that the results are based on a sufficient level of reader feedback and are less influenced by small sample sizes.

In [7]:
# Identify the author with the highest average rating
# considering only books with at least 50 ratings

query_4 = """
WITH books_with_50_ratings AS (
    SELECT
        book_id,
        COUNT(rating_id) AS rating_count
    FROM ratings
    GROUP BY book_id
    HAVING COUNT(rating_id) >= 50
)

SELECT
    a.author,
    ROUND(AVG(r.rating), 2) AS average_rating,
    COUNT(r.rating_id) AS total_ratings
FROM books AS b
JOIN books_with_50_ratings AS bw
    ON b.book_id = bw.book_id
JOIN authors AS a
    ON b.author_id = a.author_id
JOIN ratings AS r
    ON b.book_id = r.book_id
GROUP BY
    a.author
ORDER BY
    average_rating DESC
LIMIT 1;
"""

top_author = pd.read_sql(query_4, con=engine)
top_author

,author,average_rating,total_ratings
0,J.K. Rowling/Mary GrandPré,4.29,310


J.K. Rowling and Mary GrandPré achieved the highest average rating among authors whose books received at least 50 ratings, with an average rating of **4.29** based on **310 ratings**.

The minimum threshold of 50 ratings helps ensure that the result reflects consistent reader feedback rather than a small number of highly positive reviews. The large volume of ratings further reinforces the strong reception of these books among readers.

<div class="alert alert-block alert-success">
<b>Comentario del revisor (2da Iteracion)</b> <a class=“tocSkip”></a>

Perfecto! La forma en la que hagamos los filtros o se haga uso de subconsultas va a cambiar el resultado en la salida de las mismas, esta última query es la correcta y también se ve afectado el cálculo del promedio de la calificación
</div>

### Review Behavior of Highly Active Users

This analysis calculates the average number of written reviews submitted by users who rated more than 50 books. The goal is to understand whether highly active rating users also contribute written feedback.

In [8]:
# Calculate the average number of written reviews submitted by highly active users

query_5 = """
WITH active_users AS (
    SELECT
        r.username,
        COUNT(DISTINCT rv.review_id) AS user_reviews_count
    FROM ratings AS r
    LEFT JOIN reviews AS rv
        ON r.username = rv.username
    GROUP BY r.username
    HAVING COUNT(r.rating_id) > 50
)

SELECT
    ROUND(AVG(user_reviews_count), 2) AS avg_reviews_per_active_user
FROM active_users;
"""

avg_reviews_active_users = pd.read_sql(query_5, con=engine)
avg_reviews_active_users

,avg_reviews_per_active_user
0,17.46


Users who rated more than 50 books wrote an average of **17.46 text reviews**.

This suggests that highly active rating users also contribute a meaningful amount of written feedback, making them valuable users for understanding reader preferences and improving recommendation features.

## Conclusions

- The catalog contains a strong collection of modern publications, with 819 books published after January 1, 2000.
- Reader participation through written reviews is relatively limited. The most-reviewed book, Twilight, received only 7 reviews, suggesting that users are more likely to provide ratings than detailed written feedback.
- Books with higher review activity generally maintained positive ratings. Among the most-reviewed titles, Harry Potter and the Prisoner of Azkaban achieved the highest average rating (4.41).
- Penguin Books was the leading publisher, contributing 42 books with more than 50 pages.
- J.K. Rowling and Mary GrandPré achieved the highest average author rating (4.29) among books with at least 50 ratings, supported by 310 ratings.
- Highly active users wrote an average of 17.46 reviews, indicating that engaged readers contribute valuable qualitative feedback in addition to ratings.

## Recommendations
- Encourage users to submit more written reviews through incentives, badges, or engagement campaigns. Increasing review participation would provide richer qualitative feedback and improve recommendation quality.
- Prioritize highly rated books and authors within recommendation systems, featured collections, and promotional campaigns to increase user engagement and content discovery.
- Strengthen partnerships with major publishers such as Penguin Books, whose substantial catalog presence may support content expansion and reader retention initiatives.
- Develop engagement strategies targeted at highly active users, as these readers contribute both ratings and written reviews and can provide valuable insights into user preferences.
- Continue expanding the catalog of contemporary titles, as modern publications represent a significant share of the database and are likely aligned with current reader interests.